# Robustesse des CNN aux perturbations d’images — v2 (CBAM résiduel, configurations propres)

Ce notebook évalue les **six modèles déjà entraînés** : CNN classique et CNN avec CBAM, chacun avec les graines 42, 123 et 2026. Il utilise les 953 images de validation.

**Ce qui change par rapport à la v1** : les six modèles évalués sont ceux du notebook `07_stabilite_multigraines_v2` (dossier `stabilite_multigraines_v2`), et non plus ceux de la v1. Chaque architecture est évaluée avec **sa propre configuration retenue** :

| Architecture | Configuration | Origine de la graine 42 |
|---|---|---|
| CNN classique | LR 0,001 ; Dropout 0,5 | `cnn_classique_grille_v1`, configuration 2 |
| CNN + CBAM **résiduel** (v2) | LR 0,0005 ; Dropout 0,3 | `cnn_cbam_grille_v2_residuel`, configuration 3 |

Le module CBAM défini plus bas inclut la connexion résiduelle (`x + attention(x)`), indispensable pour que les poids rechargés produisent exactement les mêmes sorties qu’à l’entraînement. Les conditions de perturbation, la graine du bruit et toutes les autres règles sont **strictement identiques à la v1**, pour que les deux analyses restent comparables. Les résultats sont écrits dans un nouveau dossier `robustesse_validation_v2` ; la v1 n’est pas modifiée.

**Objectif :** mesurer l’évolution des performances lorsque l’image change légèrement. Aucun entraînement, changement de seuil ou choix d’une meilleure graine n’est effectué. La classe prédite reste celle du plus grand des deux scores bruts, comme auparavant.

| Condition | Transformation après préparation en 224 × 224 |
|---|---|
| Originale | Référence inchangée |
| Luminosité × 0,90 | Intensités multipliées par 0,90 |
| Luminosité × 1,10 | Intensités multipliées par 1,10 |
| Contraste × 0,90 | Écarts à l’intensité moyenne multipliés par 0,90 |
| Contraste × 1,10 | Écarts à l’intensité moyenne multipliés par 1,10 |
| Bruit σ = 0,01 | Bruit gaussien, écart-type 0,01 sur l’échelle [0, 1] |
| Bruit σ = 0,03 | Bruit gaussien, écart-type 0,03 sur l’échelle [0, 1] |

Les valeurs sont ensuite limitées à [0, 1]. Les transformations sont appliquées séparément, uniquement dans le rectangle occupé par l’image redimensionnée. Les bordures noires **ajoutées** par le padding restent inchangées. Ce rectangle inclut le fond et les annotations de la radiographie ; ce n’est pas un masque des poumons.

Les niveaux ci-dessus sont des choix expérimentaux fixés avant cette évaluation, pas des normes cliniques. Une seule réalisation du bruit est utilisée par image, identique pour les six modèles ; les deux niveaux utilisent le même bruit mis à l’échelle. Les écarts-types finaux décrivent uniquement les trois graines d’entraînement.

**Exécution :** ouvrir dans VS Code avec `memoire_cnn`, puis exécuter les cellules dans l’ordre. La cellule « Évaluation » effectue 42 passages de validation. Chaque passage terminé est sauvegardé et peut être réutilisé après une interruption. Ne pas lancer deux copies simultanément.

**Portée :** il s’agit d’une analyse exploratoire sur la validation déjà utilisée pour sélectionner les modèles. Le test n’est pas chargé. Le test ayant été examiné auparavant dans le projet, cette étape ne rétablit pas une évaluation finale indépendante. Résister à ces perturbations simulées ne démontre ni une robustesse à tous les appareils ni une fiabilité clinique.


In [ ]:
from pathlib import Path, PureWindowsPath
import hashlib
import json
import sys
import time
import numpy as np
import pandas as pd
import PIL
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn import __version__ as version_sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

# Si le projet change de dossier, modifier uniquement PROJET.
PROJET = Path(r"C:\Users\David KUNYIMA\OneDrive\Desktop\Memoire_CNN_Attention")
IMAGES = PROJET / "archive" / "chest_xray"
REPARTITION = PROJET / "repartitions" / "repartition_20260917_102959_713272"
FICHIER_RESULTATS = PROJET / "optimisation" / "stabilite_multigraines_v2" / "resultats_multigraines.csv"
SORTIE = PROJET / "optimisation" / "robustesse_validation_v2"
APPAREIL = torch.device("cpu")
LOT, TAILLE = 16, 224
GRAINE_BRUIT = 314159
CLASSES = {"NORMAL": 0, "PNEUMONIA": 1}
CONDITIONS = [
    {"id": "originale", "type": "identite", "valeur": 0.0, "libelle": "Originale"},
    {"id": "luminosite_090", "type": "luminosite", "valeur": 0.9, "libelle": "Luminosité × 0,90"},
    {"id": "luminosite_110", "type": "luminosite", "valeur": 1.1, "libelle": "Luminosité × 1,10"},
    {"id": "contraste_090", "type": "contraste", "valeur": 0.9, "libelle": "Contraste × 0,90"},
    {"id": "contraste_110", "type": "contraste", "valeur": 1.1, "libelle": "Contraste × 1,10"},
    {"id": "bruit_001", "type": "bruit", "valeur": 0.01, "libelle": "Bruit σ = 0,01"},
    {"id": "bruit_003", "type": "bruit", "valeur": 0.03, "libelle": "Bruit σ = 0,03"},
]
torch.use_deterministic_algorithms(True)

def exiger(condition, message):
    if not condition:
        raise RuntimeError(message)

def empreinte_fichier(chemin):
    h = hashlib.sha256()
    with open(chemin, "rb") as fichier:
        for bloc in iter(lambda: fichier.read(1024 * 1024), b""):
            h.update(bloc)
    return h.hexdigest()

def ecrire_json(obj, chemin):
    temporaire = chemin.with_suffix(".tmp")
    temporaire.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    temporaire.replace(chemin)

def ecrire_csv(tableau, chemin, **options):
    temporaire = chemin.with_suffix(".tmp")
    tableau.to_csv(temporaire, index=False, encoding="utf-8-sig", float_format="%.17g", **options)
    temporaire.replace(chemin)

print("Matériel :", APPAREIL)
display(pd.DataFrame(CONDITIONS)[["libelle", "type", "valeur"]])


## 1. Préparer les mêmes images et les perturbations

Les images préparées sont gardées en mémoire pour éviter de rouvrir 42 fois les mêmes fichiers. Les images et leurs masques occupent environ 96 Mo. Les fichiers d’origine restent inchangés.

Le contraste est centré sur la moyenne des pixels du rectangle de l’image, hors padding ajouté. Le bruit dépend de l’identifiant de l’image, et non de l’ordre des lots ou de la graine du modèle.


In [ ]:
def preparer_avec_masque(image):
    gris = image.convert("L")
    preparee = ImageOps.pad(
        gris, (TAILLE, TAILLE), method=Image.Resampling.BILINEAR,
        color=0, centering=(0.5, 0.5)
    )
    # Toute l'image source est marquée, même ses pixels initialement noirs.
    blanc = Image.new("L", gris.size, color=255)
    masque = ImageOps.pad(
        blanc, (TAILLE, TAILLE), method=Image.Resampling.NEAREST,
        color=0, centering=(0.5, 0.5)
    )
    return np.array(preparee, dtype=np.uint8), np.array(masque) > 0

def bruit_image(identifiant, forme):
    cle = f"{GRAINE_BRUIT}|{identifiant}".encode("utf-8")
    graine = int.from_bytes(hashlib.sha256(cle).digest()[:8], "little")
    generateur = np.random.Generator(np.random.PCG64(graine))
    return generateur.standard_normal(forme).astype(np.float32)

def transformer(tableau_u8, masque, condition, identifiant):
    x = tableau_u8.astype(np.float32) / 255.0
    nature, valeur = condition["type"], condition["valeur"]
    if nature == "identite":
        return x
    resultat = x.copy()
    if nature == "luminosite":
        resultat[masque] = x[masque] * np.float32(valeur)
    elif nature == "contraste":
        moyenne = x[masque].mean(dtype=np.float64)
        resultat[masque] = (x[masque] - moyenne) * valeur + moyenne
    elif nature == "bruit":
        bruit = bruit_image(identifiant, x.shape)
        resultat[masque] = x[masque] + np.float32(valeur) * bruit[masque]
    else:
        raise ValueError(f"Transformation inconnue : {nature}")
    return np.clip(resultat, 0.0, 1.0).astype(np.float32)

exiger(FICHIER_RESULTATS.is_file(), f"Fichier absent : {FICHIER_RESULTATS}")
val_df = pd.read_csv(REPARTITION / "val.csv", dtype={"groupe": str, "empreinte": str})
colonnes = {"chemin_relatif", "groupe", "empreinte", "classe"}
exiger(colonnes.issubset(val_df.columns), "Colonnes de validation manquantes.")
exiger(not val_df[list(colonnes)].isna().any().any(), "Données de validation incomplètes.")
exiger(val_df["chemin_relatif"].is_unique, "Chemins de validation dupliqués.")
exiger(val_df["classe"].value_counts().to_dict() == {"NORMAL": 269, "PNEUMONIA": 684}, "Effectifs de validation inattendus.")
REELS = val_df["classe"].map(CLASSES).to_numpy(dtype=np.int64)
IDENTIFIANTS = val_df["chemin_relatif"].tolist()
PREPAREES = np.empty((len(val_df), TAILLE, TAILLE), dtype=np.uint8)
MASQUES = np.empty_like(PREPAREES, dtype=bool)
empreinte_images = hashlib.sha256()
for i, identifiant in enumerate(tqdm(IDENTIFIANTS, desc="Préparation de la validation")):
    with Image.open(IMAGES / identifiant) as image:
        PREPAREES[i], MASQUES[i] = preparer_avec_masque(image)
    exiger(MASQUES[i].any(), "Une image ne possède aucun pixel utilisable.")
    exiger((PREPAREES[i][~MASQUES[i]] == 0).all(), "Erreur de géométrie du padding.")
    empreinte_images.update(identifiant.encode("utf-8") + b"\0")
    empreinte_images.update(PREPAREES[i].tobytes())
    empreinte_images.update(MASQUES[i].tobytes())
SHA_IMAGES = empreinte_images.hexdigest()
print("Validation :", len(val_df), "images. Mémoire images + masques :", round((PREPAREES.nbytes + MASQUES.nbytes) / 1e6, 1), "Mo")


## 2. Vérifier visuellement les transformations

Deux exemples fixes sont affichés : le premier NORMAL et le premier PNEUMONIA dans la liste de validation. Ils ne sont pas choisis en fonction des prédictions. Les étiquettes viennent du dataset.


In [ ]:
fig, axes = plt.subplots(2, len(CONDITIONS), figsize=(18, 6))
for ligne, classe in enumerate(CLASSES):
    i = int(np.flatnonzero(val_df["classe"].to_numpy() == classe)[0])
    for colonne, condition in enumerate(CONDITIONS):
        x = transformer(PREPAREES[i], MASQUES[i], condition, IDENTIFIANTS[i])
        exiger(x.dtype == np.float32 and np.isfinite(x).all(), "Valeurs de pixels incorrectes.")
        exiger(0 <= x.min() <= x.max() <= 1, "Pixels hors de [0, 1].")
        exiger((x[~MASQUES[i]] == 0).all(), "Le padding a été modifié.")
        axes[ligne, colonne].imshow(x, cmap="gray", vmin=0, vmax=1)
        axes[ligne, colonne].set_title(condition["libelle"], fontsize=10)
        axes[ligne, colonne].axis("off")
    axes[ligne, 0].set_title(f"{classe}\nOriginale", fontsize=10)
fig.suptitle("Exemples de validation — transformations appliquées séparément", fontsize=14)
fig.tight_layout()
plt.show()
print("Les bordures ajoutées sont conservées. Une copie de cette figure sera enregistrée après les contrôles du protocole.")
figure_exemples = fig


## 3. Recharger les six modèles sauvegardés

Les définitions sont identiques à celles du notebook 07 v2 (CBAM avec connexion résiduelle). Les poids sont rechargés avec vérification stricte. Le notebook contrôle les graines, les époques, les paramètres choisis et la répartition de validation.


In [ ]:
class CNNClassique(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.bloc1 = nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc2 = nn.Sequential(nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc3 = nn.Sequential(nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.reduction = nn.AdaptiveAvgPool2d((4,4))
        self.classificateur = nn.Sequential(nn.Flatten(),nn.Linear(64*4*4,64),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64,2))
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(x))

class AttentionCanaux(nn.Module):
    def __init__(self, canaux, reduction=16):
        super().__init__()
        dim=max(1,canaux//reduction)
        self.reseau_partage=nn.Sequential(
            nn.Conv2d(canaux,dim,1,bias=False),nn.ReLU(),
            nn.Conv2d(dim,canaux,1,bias=False))
    def forward(self,x):
        moyenne=x.mean(dim=(2,3),keepdim=True)
        maximum=x.amax(dim=(2,3),keepdim=True)
        return x*torch.sigmoid(self.reseau_partage(moyenne)+self.reseau_partage(maximum))

class AttentionSpatiale(nn.Module):
    def __init__(self):
        super().__init__()
        self.convolution=nn.Conv2d(2,1,7,padding=3,bias=False)
    def forward(self,x):
        resume=torch.cat([x.mean(dim=1,keepdim=True),x.amax(dim=1,keepdim=True)],dim=1)
        return x*torch.sigmoid(self.convolution(resume))

class CBAM(nn.Module):
    def __init__(self,canaux=64,reduction=16):
        super().__init__()
        self.attention_canaux=AttentionCanaux(canaux,reduction)
        self.attention_spatiale=AttentionSpatiale()
    def forward(self,x):
        # v2 : connexion résiduelle, identique aux notebooks 06 v2 et 07 v2.
        attenue = self.attention_spatiale(self.attention_canaux(x))
        return x + attenue

class CNNAvecCBAM(CNNClassique):
    def __init__(self,dropout=0.3):
        super().__init__(dropout)
        self.cbam=CBAM(64,16)
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(self.cbam(x)))


In [ ]:
ARCHITECTURES = {
    "CNN_classique": (CNNClassique, "CNNClassique_89026_v1", 89026),
    "CNN_CBAM": (CNNAvecCBAM, "CNNAvecCBAM_89636_apres_bloc3_r16_k7_residuel_v2", 89636),
}
# Configuration retenue propre à chaque architecture (voir notebook 07 v2).
CONFIGURATIONS = {
    "CNN_classique": {"lr": 0.001, "dropout": 0.5},
    "CNN_CBAM": {"lr": 0.0005, "dropout": 0.3},
}
VERSIONS = {
    "python": sys.version, "torch": str(torch.__version__), "numpy": np.__version__,
    "pandas": pd.__version__, "pillow": PIL.__version__, "sklearn": version_sklearn,
}
PRETRAITEMENT = "L; ImageOps.pad 224 bilineaire noir centre; /255; aucune augmentation"
table_modeles = pd.read_csv(FICHIER_RESULTATS)
necessaires = {"modele", "graine", "epoque_retenue", "lr", "dropout", "sauvegarde", "val_f1_macro", "val_exactitude", "val_rappel_pneumonie", "val_specificite_NORMAL"}
exiger(necessaires.issubset(table_modeles.columns), "Le tableau des modèles est incomplet.")
exiger(not table_modeles[list(necessaires)].isna().any().any(), "Une information nécessaire manque dans le tableau des modèles.")
exiger(len(table_modeles) == 6 and not table_modeles.duplicated(["modele", "graine"]).any(), "Il faut exactement six modèles distincts.")
exiger(set(zip(table_modeles["modele"], table_modeles["graine"])) == {(m, g) for m in ARCHITECTURES for g in [42, 123, 2026]}, "Couples modèle/graine inattendus.")
for nom_config, reglages in CONFIGURATIONS.items():
    sous_table = table_modeles[table_modeles["modele"] == nom_config]
    exiger(np.allclose(sous_table["lr"], reglages["lr"], rtol=0, atol=1e-12)
           and np.allclose(sous_table["dropout"], reglages["dropout"], rtol=0, atol=1e-12),
           f"Réglages différents de ceux retenus pour {nom_config}.")
table_modeles = table_modeles.sort_values(["modele", "graine"]).reset_index(drop=True)

def retrouver_sauvegarde(texte):
    # Rend le tableau utilisable si tout le projet a été déplacé.
    morceaux = PureWindowsPath(str(texte)).parts
    positions = [i for i, nom in enumerate(morceaux) if nom == "optimisation"]
    exiger(len(positions) == 1, f"Chemin de sauvegarde inattendu : {texte}")
    relatifs = morceaux[positions[0]:]
    exiger(".." not in relatifs, "Chemin relatif incorrect.")
    return PROJET.joinpath(*relatifs)

MODELES, INFORMATIONS = {}, []
SHA_VAL = empreinte_fichier(REPARTITION / "val.csv")
for ligne in table_modeles.to_dict("records"):
    nom, graine = ligne["modele"], int(ligne["graine"])
    classe, architecture, nombre_parametres = ARCHITECTURES[nom]
    chemin = retrouver_sauvegarde(ligne["sauvegarde"])
    exiger(chemin.is_file(), f"Sauvegarde introuvable : {chemin}")
    etat = torch.load(chemin, map_location="cpu", weights_only=True)
    protocole = etat["protocole"]
    exiger(protocole["architecture"] == architecture, f"Architecture différente : {nom}.")
    exiger(protocole["graine"] == graine and etat["epoque"] == int(ligne["epoque_retenue"]), "Graine ou époque incompatible.")
    exiger(protocole["sha256_csv"]["val"] == SHA_VAL, "La validation diffère de celle de l’entraînement.")
    exiger(protocole["classes"] == CLASSES and protocole["pretraitement"] == PRETRAITEMENT, "Classes ou préparation incompatibles.")
    reglages = CONFIGURATIONS[nom]
    exiger(etat["configuration"]["lr"] == reglages["lr"] and etat["configuration"]["dropout"] == reglages["dropout"], "Sauvegarde d’une autre configuration.")
    if protocole.get("versions") != VERSIONS:
        print(f"Information : versions différentes pour {nom}, graine {graine}. Les résultats originaux seront contrôlés avant les perturbations.")
    with torch.random.fork_rng(devices=[]):
        modele = classe(dropout=reglages["dropout"]).to(APPAREIL)
        modele.load_state_dict(etat["modele"], strict=True)
    exiger(sum(p.numel() for p in modele.parameters()) == nombre_parametres, "Nombre de paramètres inattendu.")
    modele.eval()
    modele.requires_grad_(False)
    MODELES[(nom, graine)] = modele
    INFORMATIONS.append({
        "modele": nom, "graine": graine, "epoque": int(etat["epoque"]),
        "lr": reglages["lr"], "dropout": reglages["dropout"],
        "architecture": architecture, "sha256_sauvegarde": empreinte_fichier(chemin),
        "sauvegarde_relative": chemin.relative_to(PROJET).as_posix(),
    })
    del etat

PROTOCOLE = {
    "version_analyse": "robustesse_validation_v2", "ensemble": "val", "effectif": len(val_df),
    "conditions": CONDITIONS, "lot": LOT, "appareil": str(APPAREIL), "classes": CLASSES,
    "pretraitement": PRETRAITEMENT, "sha256_val_csv": SHA_VAL,
    "sha256_images_preparees_et_masques": SHA_IMAGES,
    "sha256_resultats_multigraines": empreinte_fichier(FICHIER_RESULTATS),
    "modeles": INFORMATIONS, "versions": VERSIONS, "threads_torch": torch.get_num_threads(),
    "bruit": {"graine": GRAINE_BRUIT, "generateur": "PCG64", "cle": "SHA256(graine|chemin_relatif), 8 premiers octets little-endian", "realisations_par_image": 1},
    "zone_perturbee": "rectangle image redimensionnee; padding ajoute preserve",
    "contraste": "moyenne sur le rectangle image; float64 pour le calcul de cette moyenne",
    "ecretage": "[0,1] apres transformation; retour float32", "prediction": "argmax des 2 logits, NORMAL en cas d’egalite",
    "limite": "Validation deja utilisee pour la selection; test deja consulte dans le projet; perturbations simulees; trois graines d’entrainement; chaque architecture avec sa propre configuration (classique LR 0.001/Dropout 0.5, CBAM residuel LR 0.0005/Dropout 0.3).",
}
ID_PROTOCOLE = hashlib.sha256(json.dumps(PROTOCOLE, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
SORTIE.mkdir(parents=True, exist_ok=True)
(SORTIE / "predictions").mkdir(exist_ok=True)
fichier_protocole = SORTIE / "protocole.json"
if fichier_protocole.exists():
    exiger(json.loads(fichier_protocole.read_text(encoding="utf-8")) == PROTOCOLE,
           "Le protocole ou les fichiers ont changé. Conserver le dossier existant et choisir un nouveau nom SORTIE pour cette analyse.")
else:
    ecrire_json(PROTOCOLE, fichier_protocole)
figure_exemples.savefig(SORTIE / "exemples_perturbations.png", dpi=160, bbox_inches="tight")
plt.close(figure_exemples)
print("Six modèles chargés et figés. Résultats dans :", SORTIE)


## 4. Calculer les prédictions et mesurer les changements

Les colonnes `variation_*` représentent **mesure perturbée − mesure originale**, pour le même modèle et la même graine. Une valeur négative indique une dégradation. Par exemple, −0,02 en exactitude signifie une baisse de 2 points de pourcentage.

Le nombre de nouvelles erreurs et le nombre d’erreurs corrigées complètent cette comparaison. Les scores softmax enregistrés ne sont pas des probabilités cliniques calibrées.


In [ ]:
class ValidationPerturbee(Dataset):
    def __init__(self, condition):
        self.condition = condition
    def __len__(self):
        return len(val_df)
    def __getitem__(self, indice):
        x = transformer(PREPAREES[indice], MASQUES[indice], self.condition, IDENTIFIANTS[indice])
        return torch.from_numpy(x).unsqueeze(0), indice

def probabilites_pneumonie(logits):
    z = np.asarray(logits, dtype=np.float64)
    z = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z[:, 1] / exp_z.sum(axis=1)

def verifier_predictions(tableau, information, condition):
    requis = {"chemin_relatif", "groupe", "classe_vraie", "prediction", "logit_NORMAL", "logit_PNEUMONIA", "score_pneumonie", "modele", "graine", "epoque", "condition", "id_protocole"}
    exiger(requis.issubset(tableau.columns), "Fichier de prédictions incomplet.")
    exiger(len(tableau) == len(val_df) and tableau["chemin_relatif"].tolist() == IDENTIFIANTS, "Ordre ou nombre d’images incompatible.")
    exiger(tableau["groupe"].astype(str).tolist() == val_df["groupe"].tolist(), "Groupes incompatibles.")
    exiger(np.array_equal(tableau["classe_vraie"].to_numpy(), REELS), "Étiquettes incompatibles.")
    for colonne, attendu in {"modele": information["modele"], "graine": information["graine"], "epoque": information["epoque"], "condition": condition["id"], "id_protocole": ID_PROTOCOLE}.items():
        exiger((tableau[colonne] == attendu).all(), f"Prédictions incompatibles : {colonne}.")
    logits = tableau[["logit_NORMAL", "logit_PNEUMONIA"]].to_numpy(dtype=np.float64)
    exiger(np.isfinite(logits).all(), "Scores bruts non finis.")
    exiger(np.array_equal(logits.argmax(axis=1), tableau["prediction"].to_numpy()), "Classes et scores bruts incohérents.")
    exiger(np.allclose(probabilites_pneumonie(logits), tableau["score_pneumonie"], rtol=0, atol=1e-10), "Scores softmax incohérents.")

def obtenir_predictions(information, condition):
    nom, graine = information["modele"], information["graine"]
    destination = SORTIE / "predictions" / f"{nom}_graine_{graine}__{condition['id']}.csv"
    if destination.exists():
        tableau = pd.read_csv(destination, dtype={"groupe": str})
        verifier_predictions(tableau, information, condition)
        return tableau, True
    modele = MODELES[(nom, graine)]
    chargeur = DataLoader(ValidationPerturbee(condition), batch_size=LOT, shuffle=False, num_workers=0, drop_last=False)
    logits = np.empty((len(val_df), 2), dtype=np.float32)
    vus = np.zeros(len(val_df), dtype=bool)
    modele.eval()
    with torch.no_grad():
        for images, indices in tqdm(chargeur, desc=f"{nom} / {graine} / {condition['id']}", leave=False):
            scores = modele(images.to(APPAREIL))
            exiger(scores.shape == (len(indices), 2) and torch.isfinite(scores).all().item(), "Sortie du modèle incorrecte.")
            positions = indices.numpy()
            logits[positions] = scores.cpu().numpy()
            vus[positions] = True
    exiger(vus.all(), "Des images n’ont pas été évaluées.")
    tableau = val_df[["chemin_relatif", "groupe"]].copy()
    tableau["classe_vraie"] = REELS
    tableau["prediction"] = logits.argmax(axis=1)
    tableau["logit_NORMAL"], tableau["logit_PNEUMONIA"] = logits[:, 0], logits[:, 1]
    tableau["score_pneumonie"] = probabilites_pneumonie(logits)
    for cle in ["modele", "graine", "epoque"]:
        tableau[cle] = information[cle]
    tableau["condition"], tableau["id_protocole"] = condition["id"], ID_PROTOCOLE
    verifier_predictions(tableau, information, condition)
    ecrire_csv(tableau, destination)
    return tableau, False

def calculer_mesures(tableau):
    y, prediction = tableau["classe_vraie"].to_numpy(), tableau["prediction"].to_numpy()
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    return {
        "exactitude": float(accuracy_score(y, prediction)),
        "precision_pneumonie": float(precision_score(y, prediction, zero_division=0)),
        "rappel_pneumonie": float(recall_score(y, prediction, zero_division=0)),
        "specificite_NORMAL": float(tn / (tn + fp)),
        "f1_pneumonie": float(f1_score(y, prediction, zero_division=0)),
        "f1_macro": float(f1_score(y, prediction, labels=[0, 1], average="macro", zero_division=0)),
        "auc_roc": float(roc_auc_score(y, tableau["score_pneumonie"])),
        "vrais_negatifs": int(tn), "faux_positifs": int(fp),
        "faux_negatifs": int(fn), "vrais_positifs": int(tp),
    }

def comparer_predictions(reference, perturbee):
    p0, p1 = reference["prediction"].to_numpy(), perturbee["prediction"].to_numpy()
    bon0, bon1 = p0 == REELS, p1 == REELS
    nouvelles = int(np.sum(bon0 & ~bon1))
    corrigees = int(np.sum(~bon0 & bon1))
    return {
        "predictions_changees": int(np.sum(p0 != p1)),
        "taux_predictions_changees": float(np.mean(p0 != p1)),
        "nouvelles_erreurs": nouvelles, "erreurs_corrigees": corrigees,
        "fraction_correctes_devenues_incorrectes": float(nouvelles / bon0.sum()) if bon0.any() else None,
    }
print("Fonctions prêtes. La cellule suivante lance les évaluations.")


## 5. Évaluation des sept conditions

Les six évaluations originales sont réalisées en premier. Leurs mesures doivent retrouver celles de `resultats_multigraines.csv` avant que le notebook applique une perturbation. Si un contrôle échoue, conserver le message pour examiner la différence ; ne pas désactiver le contrôle.

Prévoir plusieurs dizaines de minutes selon le PC. Une interruption fait perdre uniquement le passage en cours, pas les passages terminés. Réexécuter les cellules dans l’ordre pour reprendre.


In [ ]:
REFERENCES, MESURES_ORIGINALES, lignes = {}, {}, []
total_passages = len(CONDITIONS) * len(INFORMATIONS)
METRIQUES = ["f1_macro", "exactitude", "precision_pneumonie", "rappel_pneumonie", "specificite_NORMAL", "f1_pneumonie", "auc_roc"]
for condition in CONDITIONS:
    for information in INFORMATIONS:
        cle = (information["modele"], information["graine"])
        debut = time.perf_counter()
        predictions, reutilise = obtenir_predictions(information, condition)
        mesures = calculer_mesures(predictions)
        if condition["id"] == "originale":
            ligne_initiale = table_modeles[(table_modeles["modele"] == cle[0]) & (table_modeles["graine"] == cle[1])].iloc[0]
            for metrique in ["f1_macro", "exactitude", "rappel_pneumonie", "specificite_NORMAL"]:
                attendu = float(ligne_initiale[f"val_{metrique}"])
                exiger(abs(mesures[metrique] - attendu) < 1e-10,
                       f"Résultat original différent pour {cle}, {metrique} : {mesures[metrique]:.8f} au lieu de {attendu:.8f}. Vérifier les sauvegardes et la préparation avant de continuer.")
            REFERENCES[cle] = predictions
            MESURES_ORIGINALES[cle] = mesures
        original = MESURES_ORIGINALES[cle]
        ligne = {
            "modele": cle[0], "graine": cle[1], "epoque": information["epoque"],
            "condition": condition["id"], "libelle": condition["libelle"], "n_images": len(predictions),
            **mesures, **comparer_predictions(REFERENCES[cle], predictions),
            **{f"original_{m}": original[m] for m in METRIQUES},
            **{f"variation_{m}": mesures[m] - original[m] for m in METRIQUES},
        }
        lignes.append(ligne)
        bilan = pd.DataFrame(lignes)
        ecrire_csv(bilan, SORTIE / "resultats_robustesse.csv")
        etat = "réutilisé" if reutilise else f"calculé en {(time.perf_counter() - debut):.1f} s"
        print(f"{len(lignes):02d}/{total_passages} | {cle[0]} | graine {cle[1]} | {condition['id']} | F1 {mesures['f1_macro']:.5f} | variation {ligne['variation_f1_macro']:+.5f} | {etat}")
    if condition["id"] == "originale":
        print("Les six résultats originaux correspondent au tableau précédent. Début des perturbations.")
exiger(len(bilan) == 42 and not bilan.duplicated(["modele", "graine", "condition"]).any(), "Bilan incomplet.")
print("Évaluation terminée : 42 résultats, aucun entraînement effectué.")


## 6. Résumer et comparer

Chaque condition est résumée séparément : on ne mélange pas les niveaux de bruit et les changements de contraste dans un classement global. Les variations sont calculées pour chaque graine avant d’en prendre la moyenne.

Les écarts-types utilisent trois graines (`ddof=1`). Ce ne sont pas des intervalles de confiance. Une synthèse séparée exclut la graine 42 qui a servi à sélectionner les hyperparamètres. Le nombre d’images reste 953 : leurs versions perturbées ne sont pas des observations indépendantes supplémentaires.


In [ ]:
colonnes_synthese = METRIQUES + [f"variation_{m}" for m in METRIQUES] + ["taux_predictions_changees", "fraction_correctes_devenues_incorrectes"]
def synthetiser(tableau):
    synthese = tableau.groupby(["modele", "condition", "libelle"], sort=False)[colonnes_synthese].agg(["mean", "std"])
    synthese.columns = [f"{mesure}_{stat}" for mesure, stat in synthese.columns]
    return synthese.reset_index()

synthese = synthetiser(bilan)
ecrire_csv(synthese, SORTIE / "synthese_robustesse.csv")
ecrire_csv(synthetiser(bilan[bilan["graine"] != 42]), SORTIE / "synthese_robustesse_nouvelles_graines.csv")
apparie = bilan.pivot(index=["graine", "condition"], columns="modele", values=["f1_macro", "variation_f1_macro"])
apparie.columns = [f"{mesure}_{nom}" for mesure, nom in apparie.columns]
apparie["ecart_f1_CBAM_moins_classique"] = apparie["f1_macro_CNN_CBAM"] - apparie["f1_macro_CNN_classique"]
apparie["ecart_variation_CBAM_moins_classique"] = apparie["variation_f1_macro_CNN_CBAM"] - apparie["variation_f1_macro_CNN_classique"]
ecrire_csv(apparie.reset_index(), SORTIE / "comparaison_robustesse_par_graine.csv")

display(synthese[["modele", "libelle", "f1_macro_mean", "f1_macro_std", "variation_f1_macro_mean", "rappel_pneumonie_mean", "specificite_NORMAL_mean"]].round(5))
print("Une variation négative indique une dégradation par rapport à l’image originale.")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
ordre = [c["id"] for c in CONDITIONS]
etiquettes = ["Originale", "Lumière\n× 0,90", "Lumière\n× 1,10", "Contraste\n× 0,90", "Contraste\n× 1,10", "Bruit\nσ 0,01", "Bruit\nσ 0,03"]
for axe, metrique, titre in zip(axes, ["f1_macro", "rappel_pneumonie", "specificite_NORMAL"], ["Variation du F1 macro", "Variation du rappel PNEUMONIA", "Variation de la spécificité NORMAL"]):
    for nom, decalage, couleur, label in [("CNN_classique", -0.08, "#2563eb", "CNN classique"), ("CNN_CBAM", 0.08, "#c66b14", "CNN avec CBAM")]:
        table = synthese[synthese["modele"] == nom].set_index("condition").loc[ordre]
        axe.errorbar(np.arange(len(ordre)) + decalage, table[f"variation_{metrique}_mean"], yerr=table[f"variation_{metrique}_std"], fmt="o", capsize=3, color=couleur, label=label)
    axe.axhline(0, color="gray", linewidth=1)
    axe.set_xticks(np.arange(len(ordre)), etiquettes, fontsize=8)
    axe.set_title(titre, fontsize=11)
    axe.set_ylabel("Perturbée − originale")
    axe.grid(axis="y", alpha=0.2)
axes[0].legend(fontsize=9)
fig.suptitle("Validation : moyenne et écart-type sur trois graines d’entraînement", fontsize=14)
fig.tight_layout()
fig.savefig(SORTIE / "variations_robustesse.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)
ecrire_json({"id_protocole": ID_PROTOCOLE, "passages_termines": len(bilan), "effectif_par_passage": len(val_df)}, SORTIE / "termine.json")
print("Fichier à transmettre :", SORTIE / "resultats_robustesse.csv")
print("Figure complémentaire :", SORTIE / "variations_robustesse.png")


## Lecture des résultats et limites

- Comparer à la fois le score obtenu et sa variation par rapport à l’original : un modèle peu performant peut aussi changer très peu.
- Examiner séparément le rappel PNEUMONIA et la spécificité NORMAL. Une moyenne globale peut cacher une dégradation d’une classe.
- Des différences sur ces transformations ne prouvent pas leur origine : ce protocole n’identifie pas causalement les pixels responsables et n’isole pas l’effet du padding.
- Ne pas choisir les niveaux de perturbation après avoir vu quel modèle gagne. Toute nouvelle série de transformations doit être décrite comme une nouvelle exploration.
- L’évaluation concerne des images JPEG préparées, pas une simulation physique validée des appareils de radiographie. L’écrêtage à [0, 1] peut modifier la distribution du bruit près du noir et du blanc.
- La calibration des scores, la robustesse à un autre centre et la validation clinique ne sont pas évaluées ici.

**Références des opérations utilisées :** [préparation et padding avec Pillow](https://pillow.readthedocs.io/en/stable/reference/ImageOps.html#PIL.ImageOps.pad) ; [générateurs aléatoires NumPy](https://numpy.org/doc/stable/reference/random/generator.html). Les paramètres du protocole sont nos choix expérimentaux ; ces documentations ne les valident pas sur le plan clinique.

**Vérification avant livraison (v2) :** syntaxe de toutes les cellules vérifiée ; chemins, identifiants d’architecture et configurations alignés sur les notebooks 06 v2 et 07 v2. Les modèles et les radiographies du projet restent à exécuter sur votre ordinateur ; leurs résultats ne sont pas préremplis.
